# 🔥 Fenix Core — دفتر تدريب عقل Fenix الخاص

**الخطوات:**
1. من شريط الأعلى: **Runtime (بيئة التنفيذ) ← Change runtime type (تغيير نوع البيئة) ← T4 GPU ← Save**
2. اضغط زر ▶ على كل خلية بالترتيب (انتظر كل خلية تخلص)
3. آخر خلية يعطيك **ملف fenix-core-lora.zip** — حمّله وأرسله لي

⏱ الوقت الكلي: 10–20 دقيقة تقريباً (البيانات الحالية صغيرة)

## 1️⃣ التحقق من وجود الـ GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ اختر T4 GPU من: Runtime ← Change runtime type ثم أعد تشغيل هذه الخلية"
print("✅ GPU جاهز:", torch.cuda.get_device_name(0))

## 2️⃣ جلب بيانات التدريب من GitHub — اضغط ▶ فقط، بدون رفع أي ملف

In [ ]:
import os
if os.path.exists('/content/fenix-core/training/runs') and os.listdir('/content/fenix-core/training/runs'):
    print('🛡️ في هذه الجلسة نموذج مدرَّب جاهز — تخطيت إعادة السحب حمايةً لنتيجتك')
    print('شغّل خلية 6 لحفظه في Drive، ولا تعد تشغيل هذه الخلية إلا في جلسة جديدة فارغة')
else:
    os.chdir('/content')
    os.system('rm -rf /content/fenix-core')
    os.system('git clone -q https://github.com/hakarikenji/Fenix-ai /content/fenix-core')
    if not os.path.exists('/content/fenix-core/training/train_lora.py'):
        print('⚠️ git clone فشل — أجرب الطريقة البديلة (تحميل مباشر)...')
        os.system('curl -sL https://github.com/hakarikenji/Fenix-ai/archive/refs/heads/main.tar.gz -o /content/fc.tgz')
        os.system('cd /content && tar xzf fc.tgz && mv Fenix-ai-main fenix-core && rm -f fc.tgz')
    assert os.path.exists('/content/fenix-core/training/train_lora.py'), '❌ فشل سحب المستودع بطريقتين — تحقق من اتصال الإنترنت وأعد تشغيل هذه الخلية'
    print('✅ المستودع والبيانات جاهزة — تم السحب من GitHub')


## 3️⃣ تثبيت مكتبات التدريب (~دقيقتين)

In [ ]:
%pip install -q -U peft trl bitsandbytes datasets accelerate sentencepiece
%pip install -q 'transformers==4.51.3'
import transformers, peft, datasets
print('✅ المكتبات جاهزة — transformers', transformers.__version__)

## 4️⃣ التدريب (الخلية الطويلة ~10-20 دقيقة)

سيرفع النموذج Qwen3-4B ويضيف عليه "طبقات تعلم" صغيرة (LoRA) — دون المساس بالنموذج الأصلي.
ستشاهد `train_loss` ينزل تدريجياً و `eval_loss` يُقاس كل جولة.
⚠️ إذا خلصت الخلية بسرعة وبدون أي تقدّم = التدريب ما اشتغل — اقرأ الرسالة في آخر الخلية

In [ ]:
%cd /content/fenix-core
import os
assert os.path.exists('training/train_lora.py'), '❌ السكربت غير موجود — أعد تشغيل خلية 2 (السحب من GitHub) أولاً'
!python training/train_lora.py --data-dir training/data --out training/runs
import json as _json
from pathlib import Path
if list(Path('training/runs').glob('*/run.json')):
    print()
    print('✅ التدريب اكتمل فعلاً — روح للخلية 5')
else:
    print()
    print('❌ التدريب ما اكتمل — اقرأ الرسالة الحمراء أعلاه في هذه الخلية.')
    print('أشهر سبب: ما فعّلت GPU. الحل: Runtime ← Change runtime type ← T4 GPU ← ثم Runtime ← Restart and run all')


## 5️⃣ فحص سريع — جرّب عقلك الجديد مباشرة!

In [ ]:
%cd /content/fenix-core
import json
from pathlib import Path
runs = sorted(Path('training/runs').glob('*/run.json'))
if not runs:
    print('⏭️ أُتخطت هذه الخلية — التدريب لم يكتمل بعد. الرسالة الحمراء الحقيقية موجودة في خلية 4 فوق')
else:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch
    run = runs[-1].parent
    best = json.load(open(run / 'run.json'))['best_checkpoint']
    print('best checkpoint:', best)
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')
    m = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B-Instruct-2507', device_map='auto',
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)))
    m = PeftModel.from_pretrained(m, best)
    def ask(q):
        t = tok.apply_chat_template([{'role':'system','content':'You are Fenix, an AI assistant built by Hakari.'},
                                     {'role':'user','content':q}], tokenize=False, add_generation_prompt=True)
        ids = tok(t, return_tensors='pt').to(m.device)
        out = m.generate(**ids, max_new_tokens=200, do_sample=True, temperature=0.6)
        return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print('🧪 Q1:', ask('Who are you and who built you?'))
    print()
    print('🧪 Q2:', ask('من أنت بالضبط ومن بنى you؟'))


## 6️⃣ الحفظ — حمّل ملف النموذج المدرّب

In [ ]:
import os, json, shutil, zipfile
from pathlib import Path

RUNS = Path('/content/fenix-core/training/runs')
runs = sorted(RUNS.glob('*/run.json')) if RUNS.exists() else []
if not runs:
    print('⏭️ أُتخطت هذه الخلية — لا يوجد نموذج مدرَّب بعد. أكمل التدريب (خلية 4) أولاً')
else:
    run_dir = runs[-1].parent
    manifest = json.load(open(run_dir / 'run.json'))
    best = Path(manifest['best_checkpoint'])
    print('أفضل نقطة تدريب:', best.name)

    # 1) نسخة خفيفة (النموذج فقط بدون ملفات المحسّن الضخمة) — مثالية للهاتف
    slim = Path('/content/fenix-core/slim')
    shutil.rmtree(slim, ignore_errors=True)
    shutil.copytree(best, slim / 'checkpoint')
    for junk in list(slim.rglob('*')):
        if junk.name in ('optimizer.pt', 'scheduler.pt', 'rng_state.pth'):
            junk.unlink()
    slim_zip = Path('/content/fenix-core/fenix-core-adapter.zip')
    if slim_zip.exists():
        slim_zip.unlink()
    with zipfile.ZipFile(slim_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in slim.rglob('*'):
            if f.is_file():
                z.write(f, str(f.relative_to(slim)))

    # 2) النسخة الكاملة + الخفيفة → Google Drive (آمنة ضد أي انفصال)
    full_zip = Path('/content/fenix-core/fenix-core-lora.zip')
    if not full_zip.exists():
        with zipfile.ZipFile(full_zip, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in RUNS.rglob('*'):
                if f.is_file():
                    z.write(f, str(Path('runs') / f.relative_to(RUNS)))
    from google.colab import drive
    drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/Fenix')
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(full_zip, dest / full_zip.name)
    shutil.copy2(slim_zip, dest / slim_zip.name)
    print('📁 النسختان محفوظتان في Drive → MyDrive/Fenix — لا تضيعان مهما انفصلت الجلسة')

    # 3) تحميل النسخة الخفيفة على جهازك
    from google.colab import files
    files.download(str(slim_zip))
    print('📥 أرسل لي ملف fenix-core-adapter.zip وسنوصل العقل بالتطبيق')


## 7️⃣ استعادة نتيجة سابقة من Drive — بعد أي انفصال، بدون إعادة تدريب

In [ ]:
import os, shutil, tempfile, zipfile
from pathlib import Path

bak = Path('/content/drive/MyDrive/Fenix/fenix-core-lora.zip')
if not bak.exists():
    print('⏭️ لا يوجد backup في Drive بعد — شغّل هذه الخلية فقط إذا سبق وحفظت نسخة')
else:
    assert os.path.exists('/content/fenix-core/training/train_lora.py'), 'شغّل خلية 2 (سحب المستودع) أولاً'
    tmp = Path(tempfile.mkdtemp())
    with zipfile.ZipFile(bak) as zf:
        zf.extractall(tmp)
    found = sorted(tmp.rglob('run.json'))
    assert found, 'الملف المضغوط لا يحتوي نتائج تدريب'
    dest_root = Path('/content/fenix-core/training/runs')
    dest_root.mkdir(parents=True, exist_ok=True)
    for rj in found:
        target = dest_root / rj.parent.name
        if not target.exists():
            shutil.copytree(rj.parent, target)
    print('✅ استُعيد التدريب من Drive — شغّل خلية 5 للتجربة أو خلية 6 لحفظ نسخة جديدة')
